# Hindlimb Model Edits 
+ Use this notebook to keep track of code used to edit the hindlimb model
+ Place functions in the `src` directory and import them in the blocks that need them 
+ Document your changes with markdown cells before the code block in the style of:

## Step 0: Load the model
This step loads the model and sets up the environment for editing.

In [1]:
import opensim as osim

# Load the model
model = osim.Model('rat_hindlimb.osim')

[info] Loaded model RatHindlimbRight from file rat_hindlimb.osim


## Step 1: Convert Thelen muscles to Millard
There is an error in the equilibrate muscles function for Thelen muscles, and Millard muscles are more prevalent.

In [ ]:
from src.muscle_utils import model_thelen_to_millard

millard_model = model_thelen_to_millard(model)
# Save the modified model
millard_model.printToXML('rat_hindlimb_millard.osim')

## Step 2: Optimize tendon slack lengths
This is done initially so that the model can be opened in OpenSim Creator for easier muscle editing. It will need to be redone later to account for changes to moment arms. This method is currently using the whole range of motion of each joint, but you could also specify coordinate combinations that you expect to see in the analyzed motion. Using the below method, you can catch most issues, but if you want to just focus on a specific motion, it might be unnecessary. 

In [ ]:
from src.muscle_utils import optimize_model_tsl
from src.musculoskeletal_graph import MusculoskeletalGraph

millard_tsl_model = millard_model.clone()
graph = MusculoskeletalGraph(millard_tsl_model) # How the heck does this work? It's supposed to take a path to a model...
    
muscle_lengths = graph.get_all_muscle_lengths_rom(min_points=10)
millard_tsl_model: osim.Model = optimize_model_tsl(millard_tsl_model, muscle_lengths)

millard_tsl_model.printToXML('rat_hindlimb_millard_tsl.osim')

## Step 3: Modify muscle wrapping and attachment points
Because of the hard-coded value in [OpenSim's WrapCylinder.cpp code](https://github.com/opensim-org/opensim-core/blob/f9cd558ec3ea99dda206e5e412e62e23cf19bd7e/OpenSim/Simulation/Wrap/WrapCylinder.cpp#L668):

```c++
// Each muscle segment on the surface of the cylinder should be
// 0.002 meters long. This assumes the model is in meters, of course.
int numWrapSegments = (int)(aWrapResult.wrap_path_length / 0.002);
if (numWrapSegments < 1) numWrapSegments = 1;
```
we need to modify the wrapping surfaces to prevent discontinuities in muscle lengths that appear in the range of motion of the rats during normal gait




In [ ]:
import plotly.graph_objects as go
# Plot lengths to identify problems
for muscle_name, muscle_data in muscle_lengths:
    fig = go.Figure()


### BFa 
changed wrapping surfaces to attachment points
Transclude the original and edited code snippets
#### Original
```xml
<WrapCylinder id="wrap_BFa_1" name="wrap_BFa_1" radius="0.025" length="0.1" orientation="0 0 1" location="0.0 -0.05 0.0">
    <Point id="wrap_BFa_1_p1" name="wrap_BFa_1_p1" x="-0.025" y="-0.05" z="0"/>
    <Point id="wrap_BFa_1_p2" name="wrap_BFa_1_p2" x="-0.025" y="0.05" z="0"/>
</WrapCylinder>
```

#### Edited
```xml
<WrapCylinder id="wrap_BFa_1" name="wrap_BFa_1" radius="0.025" length="0.1" orientation="0 0 1" location="0.0 -0.05 0.0">
    <Point id="wrap_BFa_1_p1" name="wrap_BFa_1_p1" x="-0.025" y="-0.05" z="0"/>
    <Point id="wrap_BFa_1_p2" name="wrap_BFa_1_p2" x="-0.025" y="0.05" z="0"/>
</WrapCylinder>
```

In [ ]:
# Plot moment arms for original and modified models


## Final Step: Create Bilateral Model

In [ ]:
# from src import mirror_hindlimb
# #TODO: Implement the mirror_hindlimb function to create a bilateral model
# bilateral_model : osim.Model = mirror_hindlimb(model)

# # Save the mirrored model
# bilateral_model.printToXML('rat_hindlimb_bilateral.osim')